# funcsearch_tsp_efficiency

## Introduction

1. **Course project**: This notebook is part of the **CS5491** course project.
2. **Workflow**: We **clone** Google's official [FunSearch](https://github.com/google-deepmind/funsearch) repository and **modify and extend** it to fit this project's experimental setup.
3. **Scope**: Implement FunSearch experiments for **TSP (Traveling Salesman Problem)** and add **efficiency** improvements (e.g., fewer wasted evaluations, better sampling strategies; details follow in later sections and code).

The sections below describe the experimental steps and implementation notes.

## Clone

Run the cell below to clone upstream [google-deepmind/funsearch](https://github.com/google-deepmind/funsearch) into **`funsearch/`** (same directory as this notebook). The folder name must be **`funsearch`** so imports like `from funsearch.implementation import …` match upstream. If you already cloned as `google_deepmind_funsearch/`, rename it: `mv google_deepmind_funsearch funsearch`. If `funsearch/` already exists, remove or rename it before cloning.

In [1]:
!pwd

/Users/yangkefan/Documents/projects/CS5491AI


In [12]:
!git clone --depth 1 https://github.com/google-deepmind/funsearch.git

Cloning into 'funsearch'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 39 (delta 5), reused 21 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 1014.55 KiB | 54.00 KiB/s, done.
Resolving deltas: 100% (5/5), done.


In [14]:
!ls -la

total 280
drwxr-xr-x@ 18 yangkefan  staff    576 Apr 10 17:18 .
drwxr-xr-x  10 yangkefan  staff    320 Mar 15 11:02 ..
-rw-r--r--@  1 yangkefan  staff  10244 Mar 16 23:52 .DS_Store
drwxr-xr-x@  3 yangkefan  staff     96 Apr 10 14:27 .cursor
-rw-r--r--@  1 yangkefan  staff    119 Apr 10 13:06 .env
-rw-r--r--@  1 yangkefan  staff    206 Mar 16 11:10 .env.example
drwxr-xr-x@ 16 yangkefan  staff    512 Apr 10 17:15 .git
-rw-r--r--@  1 yangkefan  staff    143 Apr 10 17:12 .gitignore
-rw-r--r--@  1 yangkefan  staff   8065 Mar 16 21:45 README.md
drwxr-xr-x@  3 yangkefan  staff     96 Apr 10 16:55 _funsearch_import_shim
drwxr-xr-x@  3 yangkefan  staff     96 Mar 16 15:05 data
-rw-r--r--@  1 yangkefan  staff  92173 Apr 10 17:13 funcsearch_tsp_efficiency.ipynb
drwxr-xr-x@ 13 yangkefan  staff    416 Apr 10 17:19 funsearch
drwxr-xr-x@ 13 yangkefan  staff    416 Mar 16 21:07 implementation
-rw-r--r--@  1 yangkefan  staff    110 Apr 10 15:28 requirements.txt
drwxr-xr-x@  9 yangkefan  staff    288 Ma

## Config

Configuration for experiments (paths, hyperparameters, API keys, etc.) — add cells below as needed.

In [ ]:
# Mirrors implementation/config.py — ProgramsDatabaseConfig fields (nested under Config.programs_database)
programs_database_functions_per_prompt = 2
programs_database_num_islands = 10
programs_database_reset_period = 10 * 60
programs_database_cluster_sampling_temperature_init = 0.1
programs_database_cluster_sampling_temperature_period = 50

# Config (top-level fields)
num_samplers = 1
num_evaluators = 140
samples_per_prompt = 4
iterations = 50
goal_description = (
    "maximize the size of the admissible set"
)
early_stop_patience = -1
result_dir = "result"
problem = "admissible"

progressive_eval = False
stage1_timeout = 5
stage1_score_threshold_pct = 0.0

functional_dedup = False
dedup_tier1_only = False

adaptive_sampling = False
min_samples_per_prompt = 2
max_samples_per_prompt = 4
reduce_after_no_improve = 3

weighted_island_sampling = False
island_sampling_temperature = 1.0
feedback_in_prompt = False

adaptive_temperature = False
reheat_after_no_improve = 5
reheat_temperature_multiplier = 3.0


In [ ]:
SPECIFICATION = '''
"""TSP solver using greedy construction with evolvable priority heuristic."""
import numpy as np


class funsearch:
  """Decorator stubs for FunSearch (no-op)."""
  @staticmethod
  def run(f):
    return f
  @staticmethod
  def evolve(f):
    return f


def _calc_insert_cost(D, prv, nxt, ins):
  """Insertion cost of inserting city ins between prv and nxt."""
  return D[prv, ins] + D[ins, nxt] - D[prv, nxt]


def _safe_priority_score(s):
  """Convert priority() output to float; use -inf for None/NaN so argmax skips invalid."""
  if s is None:
    return float('-inf')
  try:
    v = float(s)
    return v if np.isfinite(v) else float('-inf')
  except (TypeError, ValueError):
    return float('-inf')


def solve(instance):
  """Build a TSP tour via greedy construction with cheapest insertion.

  At each step, select the next city using priority() and insert it at the
  position that minimizes the increase in tour length.

  Args:
    instance: TSPInstance with .instance_id and .dist_matrix (n x n).

  Returns:
    Tour as list of city indices (length n).
  """
  D = instance.dist_matrix
  n = D.shape[0]

  unvisited = np.ones(n, dtype=bool)
  tour = []

  for step in range(n):
    feas_ind = np.flatnonzero(unvisited)
    if step == 0:
      city = 0
    else:
      scores = np.array([
          _safe_priority_score(priority(city_idx, tour, unvisited, D))
          for city_idx in feas_ind
      ], dtype=float)
      best_idx = np.argmax(scores)
      city = feas_ind[best_idx]

    unvisited[city] = False

    if len(tour) == 0:
      tour.append(city)
    else:
      insert_costs = np.array([
          _calc_insert_cost(D, tour[i], tour[(i + 1) % len(tour)], city)
          for i in range(len(tour))
      ])
      best_insert = np.argmin(insert_costs)
      tour.insert(best_insert + 1, city)

  tour_length = sum(
      D[tour[i], tour[(i + 1) % n]] for i in range(n)
  )
  return tour, tour_length


@funsearch.run
def evaluate(instance):
  """Returns negative tour length (higher is better for FunSearch)."""
  _, tour_length = solve(instance)
  return -float(tour_length)


@funsearch.evolve
def priority(city_idx, current_tour, unvisited_mask, dist_matrix):
  """Scores a candidate city for greedy selection.

  Baseline: nearest-neighbor heuristic (prefer city closest to the tour).
  Higher score = more preferred. Try farthest insertion, regret-based, etc.

  Args:
    city_idx: Index of the candidate city.
    current_tour: List of already visited city indices.
    unvisited_mask: Boolean array, True for unvisited cities.
    dist_matrix: n x n distance matrix.

  Returns:
    A score; higher means this city is preferred as the next to add.
  """
  min_d = np.min(dist_matrix[city_idx, current_tour])
  return -min_d
'''

## FunSearch Adaption

Adaptation is done **entirely in the notebook cells below** (no edits to course `implementation/` or to files inside the cloned **`funsearch/`** tree). Clone upstream into **`funsearch/`** (see **Clone**) so the directory layout is `funsearch/implementation/…`; the bootstrap cell only adds the repo root to `sys.path`. **Requires Python 3.10+**. Run **bootstrap**, then the **adapted classes** cell, then **run**.



In [ ]:
# Clone must live in ./funsearch/ (see Clone) so `funsearch.implementation` resolves.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "funsearch" / "implementation").is_dir():
    raise FileNotFoundError(
        "Expected funsearch/implementation/ — clone into `funsearch` or run: "
        "mv google_deepmind_funsearch funsearch"
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [ ]:
# --- Adapted classes: inherit upstream funsearch.implementation (clone in ./funsearch/)
# Source derived from course implementation/ with imports rewritten.

from funsearch.implementation import evaluator as _fs_ev

# Copyright 2023 DeepMind Technologies Limited
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#    http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ==============================================================================

"""Class for evaluating programs proposed by the Sampler."""
import ast
import logging
from collections.abc import Sequence
import copy
import multiprocessing
import textwrap
from typing import Any

from funsearch.implementation import code_manipulation
from funsearch.implementation import programs_database
from implementation.functional_dedup import FunctionalDedupCache


class _FunctionLineVisitor(ast.NodeVisitor):
  """Visitor that finds the last line number of a function with a given name."""

  def __init__(self, target_function_name: str) -> None:
    self._target_function_name: str = target_function_name
    self._function_end_line: int | None = None

  def visit_FunctionDef(self, node: Any) -> None:  # pylint: disable=invalid-name
    """Collects the end line number of the target function."""
    if node.name == self._target_function_name:
      self._function_end_line = node.end_lineno
    self.generic_visit(node)

  @property
  def function_end_line(self) -> int:
    """Line number of the final line of function `target_function_name`."""
    assert self._function_end_line is not None  # Check internal correctness.
    return self._function_end_line


def _trim_function_body(generated_code: str) -> str:
  """Extracts the body of the generated function, trimming anything after it."""
  if not generated_code:
    return ''

  logging.debug("Raw generated code:\n%s", generated_code)

  # Filter out full-line comments starting with #
  lines = generated_code.splitlines()
  lines = [line for line in lines if not line.strip().startswith('#')]
  generated_code = '\n'.join(lines)

  # Remove markdown code blocks if present
  if '```' in generated_code:
    lines = generated_code.splitlines()
    # Filter out lines that start with ```
    lines = [line for line in lines if not line.strip().startswith('```')]
    generated_code = '\n'.join(lines)
    logging.debug("Cleaned markdown code:\n%s", generated_code)

  # Try to detect if the code is a full function definition (Issue 1)
  try:
    # Attempt to parse directly without wrapper first
    direct_tree = ast.parse(generated_code)
    # Check if it contains a single FunctionDef at top level
    func_defs = [node for node in ast.walk(direct_tree) if isinstance(node, ast.FunctionDef)]
    if func_defs and len(func_defs) >= 1:
       # Use the first function found (assuming it's the intended one)
       logging.debug("Detected full function definition, extracting body...")
       target_node = func_defs[0]

       if target_node.body:
           body_start_line = target_node.body[0].lineno

           lines = generated_code.splitlines()
           pass 
  except SyntaxError:
    pass

  # Ensure the code is indented for parsing
  lines = generated_code.splitlines()
  # Find first non-empty line to check indentation
  first_line_idx = next((i for i, line in enumerate(lines) if line.strip()), None)
  
  if first_line_idx is not None:
      first_line = lines[first_line_idx]
      # Check if it looks like a function definition
      if first_line.strip().startswith('def ') and ':' in first_line:
           logging.debug("Detected function signature in first line. Attempting to strip it.")
           
           # Calculate indentation of the def
           def_indent = len(first_line) - len(first_line.lstrip())
           
           # Find where the function body ends (by indentation)
           # This is complex. Let's just try to parse it as is (it's a valid function).
           try:
               tree = ast.parse(generated_code)
               # If successful, extract body
               for node in ast.walk(tree):
                   if isinstance(node, ast.FunctionDef):
                       # Found it.
                       body_nodes = node.body
                       if body_nodes:
                           start_line = body_nodes[0].lineno - 1
                           end_line = node.end_lineno
                           body_segment = lines[start_line:end_line]
                           # Dedent this segment
                           segment_str = '\n'.join(body_segment)
                           dedented = textwrap.dedent(segment_str)
                           logging.debug("Successfully extracted body from nested function.")
                           generated_code = dedented
                           lines = generated_code.splitlines()
                           first_line_idx = next((i for i, line in enumerate(lines) if line.strip()), None)
                           if first_line_idx is not None:
                               first_line = lines[first_line_idx]
                           break
           except SyntaxError:
               logging.debug("Failed to parse potential function definition, treating as body.")

      # Re-calculate indentation after potential stripping
      initial_indent = len(first_line) - len(first_line.lstrip())
      
      # Normalize indentation: remove 'initial_indent' from all lines to align left
      if initial_indent > 0:
          normalized_lines = []
          for i, line in enumerate(lines):
              if not line.strip():
                  normalized_lines.append('')
                  continue
              
              current_indent = len(line) - len(line.lstrip())
              if current_indent >= initial_indent:
                  normalized_lines.append(line[initial_indent:])
              else:
                  # Line is less indented than start (e.g. outlier), strip it to 0
                  normalized_lines.append(line.lstrip())
          generated_code = '\n'.join(normalized_lines)

  code = f'def fake_function_header():\n' + textwrap.indent(generated_code, '  ')
  
  logging.debug("Wrapped code for parsing:\n%s", code)

  tree = None
  
  # Helper to try parsing
  def _attempt_parse(code_str):
      try:
          return ast.parse(code_str)
      except SyntaxError:
          return None

  # Try 1: Parse original wrapped code
  tree = _attempt_parse(code)
  
  # Try 2: If failed, try to fix inconsistent indentation (replace 4 spaces with 2 spaces)
  if tree is None:
      logging.debug("Parse failed, attempting indentation fix...")
      
      # Strategy: Quantize indentation to multiples of 2
      fixed_lines = []
      for line in generated_code.splitlines():
          stripped = line.lstrip()
          if not stripped:
              fixed_lines.append(line)
              continue
              
          indent = len(line) - len(stripped)
          
          if indent > 0 and indent % 4 == 0:
              new_indent = indent // 2
              fixed_lines.append(' ' * new_indent + stripped)
          else:
              fixed_lines.append(line)
      
      fixed_generated_code = '\n'.join(fixed_lines)
      
      # Re-wrap
      fixed_code = f'def fake_function_header():\n' + textwrap.indent(fixed_generated_code, '  ')
          
      tree = _attempt_parse(fixed_code)
      if tree is not None:
          code = fixed_code
          logging.debug("Indentation fix successful (4->2)")

  # Try 3: Repair loop for specific lines (Issue 2)
  if tree is None:
      logging.debug("Parse failed, entering repair loop...")
      current_code = code
      max_repairs = 5
      for attempt in range(max_repairs):
          try:
              tree = ast.parse(current_code)
              code = current_code
              logging.debug("Repair successful on attempt %d", attempt)
              break
          except SyntaxError as e:
              # Try to fix the specific line
              lineno = e.lineno
              if lineno is None: 
                  break
              
              lines = current_code.splitlines()
              if lineno > len(lines):
                  break
                  
              bad_line_idx = lineno - 1
              bad_line = lines[bad_line_idx]
              
              logging.debug("SyntaxError at line %s: %s", lineno, bad_line.strip())
              
              # Heuristic: Dedent the line if it looks like control flow
              # (e.g. 'if', 'else', 'elif', 'return', 'for') that might be mis-indented
              stripped = bad_line.lstrip()
              indent = len(bad_line) - len(stripped)
              
              if indent >= 2:
                  # Try dedenting by 2 spaces
                  new_line = ' ' * (indent - 2) + stripped
                  lines[bad_line_idx] = new_line
                  current_code = '\n'.join(lines)
                  logging.debug("Attempting to dedent line %s", lineno)
              else:
                  # Can't dedent further, break to truncation
                  break

  # We keep trying and deleting code from the end until the parser succeeds.

  while tree is None:
    try:
      tree = ast.parse(code)
    except SyntaxError as e:
      logging.debug("SyntaxError at line %s, truncating...", e.lineno)
      code = '\n'.join(code.splitlines()[:e.lineno - 1])
  if not code:
    # Nothing could be saved from `generated_code`
    logging.debug("Code became empty after truncation")
    return ''

  visitor = _FunctionLineVisitor('fake_function_header')
  visitor.visit(tree)
  logging.debug("Function end line detected: %s", visitor.function_end_line)
  
  body_lines = code.splitlines()[1:visitor.function_end_line]
  logging.debug("Extracted body lines:\n%s", body_lines)
  
  # Normalize indentation to 2 spaces
  body = '\n'.join(body_lines)
  body = textwrap.dedent(body)
  body = textwrap.indent(body, '  ')
  
  return body + '\n\n'


def _sample_to_program(
    generated_code: str,
    version_generated: int | None,
    template: code_manipulation.Program,
    function_to_evolve: str,
) -> tuple[code_manipulation.Function, str]:
  """Returns the compiled generated function and the full runnable program."""
  body = _trim_function_body(generated_code)
  if version_generated is not None:
    body = code_manipulation.rename_function_calls(
        body,
        f'{function_to_evolve}_v{version_generated}',
        function_to_evolve)

  program = copy.deepcopy(template)
  evolved_function = program.get_function(function_to_evolve)
  evolved_function.body = body
  return evolved_function, str(program)


def _run_in_subprocess(program: str, function_name: str, test_input: Any, queue: multiprocessing.Queue):
  """Executes the generated code in a subprocess."""
  try:
    # Use the same dictionary for globals and locals to ensure that functions
    # defined in `program` can access imports defined in `program`.
    local_scope = {}
    # exec() executes the program in the local_scope.
    exec(program, local_scope, local_scope)
    if function_name not in local_scope:
      queue.put((None, False))
      return
    func = local_scope[function_name]
    try:
      result = func(test_input)
      if isinstance(result, (list, tuple)) or hasattr(result, '__len__') or hasattr(result, 'shape'):
          # If result is array-like, check if it's empty or handle ambiguous truth value
          # Assuming we expect a scalar score, try to convert or summarize
          import numpy as np
          if isinstance(result, np.ndarray):
             if result.size == 1:
                 result = result.item()
             else:
                 try:
                    result = float(result)
                 except:
                    # Fallback for array output when scalar expected
                    pass
    except ValueError as e:
      if "The truth value of an array" in str(e):
         logging.debug("Caught ValueError in user code: %s", e)
         result = None
      else:
         raise e
         
    queue.put((result, True))
  except Exception:  # pylint: disable=broad-except
    # Print exception for debugging purposes
    import traceback
    traceback.print_exc()
    queue.put((None, False))


class MultiprocessingSandbox(_fs_ev.Sandbox):
  """Sandbox for executing generated code."""

  def run(
      self,
      program: str,
      function_to_run: str,
      test_input: Any,
      timeout_seconds: int,
  ) -> tuple[Any, bool]:
    """Returns `function_to_run(test_input)` and whether execution succeeded."""
    queue = multiprocessing.Queue()
    process = multiprocessing.Process(
        target=_run_in_subprocess,
        args=(program, function_to_run, test_input, queue),
    )
    process.start()
    try:
      result, success = queue.get(timeout=timeout_seconds)
      process.join()
      return result, success
    except Exception:  # pylint: disable=broad-except
      # If the process times out or fails for other reasons.
      if process.is_alive():
        process.terminate()
        process.join()
      return None, False


def _calls_ancestor(program: str, function_to_evolve: str) -> bool:
  """Returns whether the generated function is calling an earlier version."""
  for name in code_manipulation.get_functions_called(program):
    if name.startswith(f'{function_to_evolve}_v'):
      return True
  return False


def _get_smallest_instance(inputs: Sequence[Any], problem: str) -> Any | None:
  """Return the smallest test instance for Tier 2 dedup (TSP: fewest cities)."""
  if not inputs:
    return None
  if problem == 'tsp':
    return min(inputs, key=lambda x: getattr(x, 'dist_matrix', x).shape[0])
  return inputs[0]


class AdaptedEvaluator(_fs_ev.Evaluator):
  """Class that analyses functions generated by LLMs."""

  def __init__(
      self,
      database: AdaptedProgramsDatabase,
      template: code_manipulation.Program,
      function_to_evolve: str,
      function_to_run: str,
      inputs: Sequence[Any],
      timeout_seconds: int = 30,
      config: Any = None,
      dedup_cache: FunctionalDedupCache | None = None,
  ):
    self._database = database
    self._template = template
    self._function_to_evolve = function_to_evolve
    self._function_to_run = function_to_run
    self._inputs = inputs
    self._timeout_seconds = timeout_seconds
    self._sandbox = MultiprocessingSandbox()
    self._config = config
    self._dedup_cache = dedup_cache

  def analyse(
      self,
      sample: str,
      island_id: int | None,
      version_generated: int | None,
  ) -> None:
    """Compiles the sample into a program and executes it on test inputs."""
    self._database.record_sample_attempted()
    try:
      new_function, program = _sample_to_program(
          sample, version_generated, self._template, self._function_to_evolve)
    except Exception:
      self._database.record_skip('Parse error')
      raise
    body = new_function.body

    if self._config and self._dedup_cache and getattr(
        self._config, 'functional_dedup', False
    ):
      if self._dedup_cache.check_tier1(body):
        logging.debug('Functional dedup Tier 1: skipping duplicate code')
        self._database.record_skip('Duplicate (Tier 1)')
        if getattr(self._config, 'feedback_in_prompt', False):
          self._database.record_failure(body, 'Duplicate (Tier 1)')
        return
      if (
          not getattr(self._config, 'dedup_tier1_only', False)
          and getattr(self._config, 'problem', '') == 'tsp'
      ):
        smallest = _get_smallest_instance(self._inputs, self._config.problem)
        if smallest is not None:
          solve_result, runs_ok = self._sandbox.run(
              program, 'solve', smallest,
              getattr(self._config, 'stage1_timeout', 5) or 5,
          )
          if runs_ok and solve_result is not None and isinstance(solve_result, (list, tuple)):
            tour = solve_result[0] if len(solve_result) >= 1 else solve_result
            tour_tuple = tuple(int(x) for x in tour)
            instance_id = getattr(smallest, 'instance_id', str(smallest))
            if self._dedup_cache.check_tier2(instance_id, tour_tuple):
              logging.debug('Functional dedup Tier 2: skipping duplicate tour')
              self._database.record_skip('Duplicate (Tier 2)')
              if getattr(self._config, 'feedback_in_prompt', False):
                self._database.record_failure(body, 'Duplicate (Tier 2)')
              return
            tier2_tour = (instance_id, tour_tuple)
          else:
            tier2_tour = None
        else:
          tier2_tour = None
      else:
        tier2_tour = None
    else:
      tier2_tour = None

    progressive = (
        self._config is not None and getattr(self._config, 'progressive_eval', False)
    )
    stage1_timeout = (
        getattr(self._config, 'stage1_timeout', 5) if progressive else None
    )
    stage1_threshold = (
        getattr(self._config, 'stage1_score_threshold_pct', 0.0) if progressive else None
    )

    if progressive and stage1_timeout is not None and self._inputs:
      smallest = _get_smallest_instance(self._inputs, getattr(self._config, 'problem', ''))
      test_output, runs_ok = self._sandbox.run(
          program, self._function_to_run, smallest, stage1_timeout)
      if not runs_ok or test_output is None:
        self._database.record_skip('Stage1 failed')
        if getattr(self._config, 'feedback_in_prompt', False):
          self._database.record_failure(body, 'Stage1 failed')
        return
      if _calls_ancestor(program, self._function_to_evolve):
        return
      if not isinstance(test_output, (int, float)):
        raise ValueError('@function.run did not return an int/float score.')
      if (
          stage1_threshold is not None
          and stage1_threshold > 0
          and island_id is not None
      ):
        worst = self._database._best_score_per_island[island_id]
        if worst > -float('inf') and test_output < worst * (1 - stage1_threshold / 100):
          self._database.record_skip('Score too low')
          if getattr(self._config, 'feedback_in_prompt', False):
            self._database.record_failure(body, 'Score too low')
          return

    scores_per_test = {}
    for current_input in self._inputs:
      test_output, runs_ok = self._sandbox.run(
          program, self._function_to_run, current_input, self._timeout_seconds)
      if not runs_ok or test_output is None:
        self._database.record_skip('Execution failed')
        if self._config and getattr(self._config, 'feedback_in_prompt', False):
          self._database.record_failure(body, 'Execution failed')
        return
      if _calls_ancestor(program, self._function_to_evolve):
        return
      if not isinstance(test_output, (int, float)):
        raise ValueError('@function.run did not return an int/float score.')
      scores_per_test[current_input] = test_output

    if scores_per_test:
      self._database.record_full_eval()
      self._database.register_program(new_function, island_id, scores_per_test)
      if self._dedup_cache and self._config and getattr(
          self._config, 'functional_dedup', False
      ):
        self._dedup_cache.add_tier1(body)
        if tier2_tour is not None:
          self._dedup_cache.add_tier2(tier2_tour[0], tier2_tour[1])


import funsearch.implementation.programs_database as _fs_pd

# Copyright 2023 DeepMind Technologies Limited
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#    http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ==============================================================================

"""A programs database that implements the evolutionary algorithm."""
from collections.abc import Mapping, Sequence
import copy
import dataclasses
import time
from typing import Any

from absl import logging
import numpy as np
import scipy

from funsearch.implementation import code_manipulation
from funsearch.implementation import config as config_lib

Signature = tuple[float, ...]
ScoresPerTest = Mapping[Any, float]


def _softmax(logits: np.ndarray, temperature: float) -> np.ndarray:
  """Returns the tempered softmax of 1D finite `logits`."""
  if not np.all(np.isfinite(logits)):
    non_finites = set(logits[~np.isfinite(logits)])
    raise ValueError(f'`logits` contains non-finite value(s): {non_finites}')
  if not np.issubdtype(logits.dtype, np.floating):
    logits = np.array(logits, dtype=np.float32)

  result = scipy.special.softmax(logits / temperature, axis=-1)
  # Ensure that probabilities sum to 1 to prevent error in `np.random.choice`.
  index = np.argmax(result)
  result[index] = 1 - np.sum(result[0:index]) - np.sum(result[index+1:])
  return result


def _reduce_score(scores_per_test: ScoresPerTest) -> float:
  """Reduces per-test scores into a single score."""
  # return scores_per_test[list(scores_per_test.keys())[-1]]
  return sum(scores_per_test.values()) / len(scores_per_test)


def _get_signature(scores_per_test: ScoresPerTest) -> Signature:
  """Represents test scores as a canonical signature."""
  return tuple(scores_per_test[k] for k in sorted(scores_per_test.keys()))


@dataclasses.dataclass(frozen=True)
class Prompt:
  """A prompt produced by the ProgramsDatabase, to be sent to Samplers.

  Attributes:
    code: The prompt, ending with the header of the function to be completed.
    version_generated: The function to be completed is `_v{version_generated}`.
    island_id: Identifier of the island that produced the implementations
       included in the prompt. Used to direct the newly generated implementation
       into the same island.
  """
  code: str
  version_generated: int
  island_id: int


class AdaptedProgramsDatabase(_fs_pd.ProgramsDatabase):
  """A collection of programs, organized as islands."""

  def __init__(
      self,
      config: config_lib.ProgramsDatabaseConfig,
      template: code_manipulation.Program,
      function_to_evolve: str,
      config_full: Any = None,
  ) -> None:
    self._config: config_lib.ProgramsDatabaseConfig = config
    self._template: code_manipulation.Program = template
    self._function_to_evolve: str = function_to_evolve
    self._config_full: Any = config_full

    # Initialize empty islands.
    self._islands: list[Island] = []
    for _ in range(config.num_islands):
      self._islands.append(
          AdaptedIsland(template, function_to_evolve, config.functions_per_prompt,
                 config.cluster_sampling_temperature_init,
                 config.cluster_sampling_temperature_period))
    self._best_score_per_island: list[float] = (
        [-float('inf')] * config.num_islands)
    self._best_program_per_island: list[code_manipulation.Function | None] = (
        [None] * config.num_islands)
    self._best_scores_per_test_per_island: list[ScoresPerTest | None] = (
        [None] * config.num_islands)

    self._last_reset_time: float = time.time()
    self._reset_count: int = 0
    self._recent_failures: list[tuple[str, str]] = []
    self._no_improve_count: int = 0

    # Efficiency stats for sample-efficient reporting
    self._eff_samples_attempted: int = 0
    self._eff_full_evaluations: int = 0
    self._eff_skipped: dict[str, int] = {}

  def record_sample_attempted(self) -> None:
    """Record that a sample was sent to the evaluator."""
    self._eff_samples_attempted += 1

  def record_skip(self, reason: str) -> None:
    """Record that a sample was skipped (early return) for the given reason."""
    self._eff_skipped[reason] = self._eff_skipped.get(reason, 0) + 1

  def record_full_eval(self) -> None:
    """Record that a sample completed full evaluation and was registered."""
    self._eff_full_evaluations += 1

  def get_efficiency_stats(self) -> dict[str, Any]:
    """Return a read-only copy of efficiency statistics."""
    return {
        "samples_attempted": self._eff_samples_attempted,
        "full_evaluations": self._eff_full_evaluations,
        "skipped": dict(self._eff_skipped),
    }

  def notify_no_improve(self, count: int) -> None:
    """Called by Sampler each iteration to update the no-improvement counter."""
    self._no_improve_count = count

  def record_failure(self, code_snippet: str, reason: str) -> None:
    """Record a rejected sample for feedback-in-prompt."""
    flat = code_snippet.replace('\n', ' ').strip()
    snippet = (flat[:200] + "...") if len(flat) > 200 else flat
    self._recent_failures.append((snippet, reason))
    if len(self._recent_failures) > 2:
      self._recent_failures.pop(0)

  def get_prompt(self) -> Prompt:
    """Returns a prompt containing implementations from one chosen island."""
    if (
        self._config_full is not None
        and getattr(self._config_full, 'weighted_island_sampling', False)
    ):
      scores = np.array(self._best_score_per_island, dtype=np.float32)
      scores = np.where(np.isfinite(scores), scores, -1e9)
      island_temp = float(getattr(self._config_full, 'island_sampling_temperature', 1.0))
      probs = _softmax(scores, island_temp)
      island_id = int(np.random.choice(len(self._islands), p=probs))
    else:
      island_id = np.random.randint(len(self._islands))
    # Compute adaptive temperature multiplier for cluster sampling
    temp_multiplier = 1.0
    if (
        self._config_full is not None
        and getattr(self._config_full, 'adaptive_temperature', False)
    ):
      reheat_threshold = int(getattr(self._config_full, 'reheat_after_no_improve', 5))
      reheat_mult = float(getattr(self._config_full, 'reheat_temperature_multiplier', 3.0))
      if self._no_improve_count >= reheat_threshold:
        temp_multiplier = reheat_mult

    code, version_generated = self._islands[island_id].get_prompt(temp_multiplier)
    if (
        self._config_full is not None
        and getattr(self._config_full, 'feedback_in_prompt', False)
        and self._recent_failures
    ):
      feedback_lines = []
      for snippet, reason in self._recent_failures[-2:]:
        short = (snippet[:100] + "...") if len(snippet) > 100 else snippet
        feedback_lines.append(f"# Rejected (reason: {reason}):\n# {short}")
      code = code + "\n\n" + "\n".join(feedback_lines)
    return Prompt(code, version_generated, island_id)

  def get_global_best_score(self) -> float:
    """Returns the current global best score across all islands."""
    return max(self._best_score_per_island)

  def get_global_best_program(self) -> tuple[code_manipulation.Function, int] | None:
    """Returns (best_program, island_id) or None if no valid program yet."""
    best_score = self.get_global_best_score()
    if best_score == -float('inf'):
      return None
    for island_id, score in enumerate(self._best_score_per_island):
      if score == best_score and self._best_program_per_island[island_id] is not None:
        return self._best_program_per_island[island_id], island_id
    return None

  def _register_program_in_island(
      self,
      program: code_manipulation.Function,
      island_id: int,
      scores_per_test: ScoresPerTest,
  ) -> None:
    """Registers `program` in the specified island."""
    self._islands[island_id].register_program(program, scores_per_test)
    score = _reduce_score(scores_per_test)
    if score > self._best_score_per_island[island_id]:
      self._best_program_per_island[island_id] = program
      self._best_scores_per_test_per_island[island_id] = scores_per_test
      self._best_score_per_island[island_id] = score
      logging.info('Best score of island %d increased to %s', island_id, score)

  def register_program(
      self,
      program: code_manipulation.Function,
      island_id: int | None,
      scores_per_test: ScoresPerTest,
  ) -> None:
    """Registers `program` in the database."""
    # In an asynchronous implementation we should consider the possibility of
    # registering a program on an island that had been reset after the prompt
    # was generated. Leaving that out here for simplicity.
    if island_id is None:
      # This is a program added at the beginning, so adding it to all islands.
      for island_id in range(len(self._islands)):
        self._register_program_in_island(program, island_id, scores_per_test)
    else:
      self._register_program_in_island(program, island_id, scores_per_test)

    # Check whether it is time to reset an island.
    if (time.time() - self._last_reset_time > self._config.reset_period):
      self._last_reset_time = time.time()
      self.reset_islands()

  def reset_islands(self) -> None:
    """Resets the weaker half of islands."""
    self._reset_count += 1
    # We sort best scores after adding minor noise to break ties.
    indices_sorted_by_score: np.ndarray = np.argsort(
        self._best_score_per_island +
        np.random.randn(len(self._best_score_per_island)) * 1e-6)
    num_islands_to_reset = self._config.num_islands // 2
    reset_islands_ids = indices_sorted_by_score[:num_islands_to_reset]
    keep_islands_ids = indices_sorted_by_score[num_islands_to_reset:]
    for island_id in reset_islands_ids:
      self._islands[island_id] = AdaptedIsland(
          self._template,
          self._function_to_evolve,
          self._config.functions_per_prompt,
          self._config.cluster_sampling_temperature_init,
          self._config.cluster_sampling_temperature_period)
      self._best_score_per_island[island_id] = -float('inf')
      founder_island_id = np.random.choice(keep_islands_ids)
      founder = self._best_program_per_island[founder_island_id]
      founder_scores = self._best_scores_per_test_per_island[founder_island_id]
      self._register_program_in_island(founder, island_id, founder_scores)


class AdaptedIsland(_fs_pd.Island):
  """A sub-population of the programs database."""

  def __init__(
      self,
      template: code_manipulation.Program,
      function_to_evolve: str,
      functions_per_prompt: int,
      cluster_sampling_temperature_init: float,
      cluster_sampling_temperature_period: int,
  ) -> None:
    self._template: code_manipulation.Program = template
    self._function_to_evolve: str = function_to_evolve
    self._functions_per_prompt: int = functions_per_prompt
    self._cluster_sampling_temperature_init = cluster_sampling_temperature_init
    self._cluster_sampling_temperature_period = (
        cluster_sampling_temperature_period)

    self._clusters: dict[Signature, Cluster] = {}
    self._num_programs: int = 0

  def register_program(
      self,
      program: code_manipulation.Function,
      scores_per_test: ScoresPerTest,
  ) -> None:
    """Stores a program on this island, in its appropriate cluster."""
    signature = _get_signature(scores_per_test)
    if signature not in self._clusters:
      score = _reduce_score(scores_per_test)
      self._clusters[signature] = AdaptedCluster(score, program)
    else:
      self._clusters[signature].register_program(program)
    self._num_programs += 1

  def get_prompt(self, temperature_multiplier: float = 1.0) -> tuple[str, int]:
    """Constructs a prompt containing functions from this island."""
    signatures = list(self._clusters.keys())
    cluster_scores = np.array(
        [self._clusters[signature].score for signature in signatures])

    # Convert scores to probabilities using softmax with temperature schedule.
    # temperature_multiplier > 1 reheats (more exploration) when stuck.
    period = self._cluster_sampling_temperature_period
    temperature = self._cluster_sampling_temperature_init * (
        1 - (self._num_programs % period) / period) * temperature_multiplier
    probabilities = _softmax(cluster_scores, temperature)

    # At the beginning of an experiment when we have few clusters, place fewer
    # programs into the prompt.
    functions_per_prompt = min(len(self._clusters), self._functions_per_prompt) #**

    idx = np.random.choice(
        len(signatures), size=functions_per_prompt, p=probabilities)
    chosen_signatures = [signatures[i] for i in idx]
    implementations = []
    scores = []
    for signature in chosen_signatures:
      cluster = self._clusters[signature]
      implementations.append(cluster.sample_program())
      scores.append(cluster.score)

    indices = np.argsort(scores)
    sorted_implementations = [implementations[i] for i in indices]
    version_generated = len(sorted_implementations) + 1
    return self._generate_prompt(sorted_implementations), version_generated

  def _generate_prompt(
      self,
      implementations: Sequence[code_manipulation.Function]) -> str:
    """Creates a prompt containing a sequence of function `implementations`."""
    implementations = copy.deepcopy(implementations)  # We will mutate these.

    # Format the names and docstrings of functions to be included in the prompt.
    versioned_functions: list[code_manipulation.Function] = []
    for i, implementation in enumerate(implementations):
      new_function_name = f'{self._function_to_evolve}_v{i}'
      implementation.name = new_function_name
      # Update the docstring for all subsequent functions after `_v0`.
      if i >= 1:
        implementation.docstring = (
            f'Improved version of `{self._function_to_evolve}_v{i - 1}`.')
      # If the function is recursive, replace calls to itself with its new name.
      implementation = code_manipulation.rename_function_calls(
          str(implementation), self._function_to_evolve, new_function_name)
      versioned_functions.append(
          code_manipulation.text_to_function(implementation))

    # Create the header of the function to be generated by the LLM.
    next_version = len(implementations)
    new_function_name = f'{self._function_to_evolve}_v{next_version}'
    header = dataclasses.replace(
        implementations[-1],
        name=new_function_name,
        body='',
        docstring=('Improved version of '
                   f'`{self._function_to_evolve}_v{next_version - 1}`.'),
    )
    versioned_functions.append(header)

    # Replace functions in the template with the list constructed here.
    prompt = dataclasses.replace(self._template, functions=versioned_functions)
    return str(prompt)


class AdaptedCluster(_fs_pd.Cluster):
  """A cluster of programs on the same island and with the same Signature."""

  def __init__(self, score: float, implementation: code_manipulation.Function):
    self._score = score
    self._programs: list[code_manipulation.Function] = [implementation]
    self._lengths: list[int] = [len(str(implementation))]

  @property
  def score(self) -> float:
    """Reduced score of the signature that this cluster represents."""
    return self._score

  def register_program(self, program: code_manipulation.Function) -> None:
    """Adds `program` to the cluster."""
    self._programs.append(program)
    self._lengths.append(len(str(program)))

  def sample_program(self) -> code_manipulation.Function:
    """Samples a program, giving higher probability to shorther programs."""
    normalized_lengths = (np.array(self._lengths) - min(self._lengths)) / (
        max(self._lengths) + 1e-6)
    # Use higher temperature (e.g. 4.0) to reduce length penalty
    probabilities = _softmax(-normalized_lengths, temperature=4.0)
    return np.random.choice(self._programs, p=probabilities)


import funsearch.implementation.sampler as _fs_samp

# Copyright 2023 DeepMind Technologies Limited
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#    http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ==============================================================================

"""Class for sampling new programs."""
import copy
from funsearch.implementation.programs_database import Island
import json
import logging
import os
import time
from collections.abc import Collection, Sequence
from pathlib import Path
from typing import Any

import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

from funsearch.implementation import code_manipulation
from funsearch.implementation import evaluator as _fs_ev_mod
from funsearch.implementation import programs_database

# Configure logging: default INFO, set to DEBUG to see prompts
logging.basicConfig(level=logging.INFO)


class AdaptedLLM(_fs_samp.LLM):
  """Language model that predicts continuation of provided source code."""

  _total_tokens_used = 0  # Class variable to persist across instances

  def __init__(
      self,
      samples_per_prompt: int,
      goal_description: str = "maximize the size of the admissible set",
  ) -> None:
    self._samples_per_prompt = samples_per_prompt
    self._goal_description = goal_description
    self.client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
    )

  @property
  def total_tokens_used(self):
      return LLM._total_tokens_used

  def _draw_sample(self, prompt: str) -> str:
    """Returns a predicted continuation of `prompt`."""
    model = os.getenv("LLM_MODEL", "arcee-ai/trinity-large-preview:free")
    user_prompt = (
        f"Please provide the implementation for the function body of `priority` "
        f"in the following code. The goal is to {self._goal_description}.\n\n{prompt}"
    )
    logging.debug("Prompt sent to LLM:\n---\n%s\n---", user_prompt)
    
    retries = 5
    for attempt in range(retries):
        try:
            resp = self.client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are a search algorithm engineer. Your goal is to improve the computational logic of the function body. STRICTLY adhere to the function's signature and return type. DO NOT change the function's category or purpose. Write concise, high-performance code using branching structures or loops if necessary. Output code only, STRICTLY NO MARKDOWN and NO COMMENTS USING '#'. Your response should contain ONLY the function body code. Do not repeat the function signature or docstring."},
                    {"role": "user", "content": user_prompt}
                ],
            )
            if resp.usage:
                LLM._total_tokens_used += resp.usage.total_tokens
            return resp.choices[0].message.content
        except Exception as e:
            print(f"LLM API call failed (attempt {attempt+1}/{retries}): {e}")
            if attempt < retries - 1:
                time.sleep(2)
            else:
                print("Max retries reached. Stopping execution.")
                raise e

  def draw_samples(self, prompt: str, n: int | None = None) -> Collection[str]:
    """Returns multiple predicted continuations of `prompt`."""
    count = n if n is not None else self._samples_per_prompt
    return [self._draw_sample(prompt) for _ in range(count)]


class AdaptedSampler(_fs_samp.Sampler):
  """Node that samples program continuations and sends them for analysis."""

  def __init__(
      self,
      database: AdaptedProgramsDatabase,
      evaluators: Sequence[AdaptedEvaluator],
      samples_per_prompt: int,
      max_iterations: int = -1,
      goal_description: str = "maximize the size of the admissible set",
      early_stop_patience: int = -1,
      result_dir: str = "result",
      problem: str = "admissible",
      template: code_manipulation.Program | None = None,
      function_to_evolve: str = "",
      config: Any = None,
  ) -> None:
    self._database = database
    self._evaluators = evaluators
    self._llm = AdaptedLLM(samples_per_prompt, goal_description)
    self._max_iterations = max_iterations
    self._early_stop_patience = early_stop_patience
    self._result_dir = result_dir
    self._problem = problem
    self._template = template
    self._function_to_evolve = function_to_evolve
    self._config = config
    self._base_samples_per_prompt = samples_per_prompt
    self._start_time = time.time()
    self._global_best_history: list[float] = []
    self._per_instance_score_history: list[dict[str, float]] = []
    self._per_instance_gap_history: list[dict[str, float]] = []
    self._last_improvement_iter = 0
    self._total_iterations = 0

  def _get_adaptive_samples_count(self, iteration: int) -> int:
    """Return samples_per_prompt, possibly reduced when no improvement."""
    if not self._config or not getattr(self._config, 'adaptive_sampling', False):
      return self._base_samples_per_prompt
    no_improve = iteration - self._last_improvement_iter
    threshold = getattr(self._config, 'reduce_after_no_improve', 3)
    if no_improve >= threshold:
      return getattr(self._config, 'min_samples_per_prompt', 2)
    return min(
        getattr(self._config, 'max_samples_per_prompt', 4),
        self._base_samples_per_prompt,
    )

  def sample(self) -> None:
    """Continuously gets prompts, samples programs, sends them for analysis."""
    iteration = 0
    while self._max_iterations == -1 or iteration < self._max_iterations:
      prompt = self._database.get_prompt()
      n_samples = self._get_adaptive_samples_count(iteration)
      samples = self._llm.draw_samples(prompt.code, n=n_samples)
      # This loop can be executed in parallel on remote evaluator machines.
      for sample in samples:
        chosen_evaluator = np.random.choice(self._evaluators)
        chosen_evaluator.analyse(
            sample, prompt.island_id, prompt.version_generated)
      
      iteration += 1
      self._total_iterations = iteration
      
      best_score = self._database._best_score_per_island[prompt.island_id]
      global_best_score = self._database.get_global_best_score()
      
      # Track improvement for early stopping
      if not self._global_best_history or global_best_score > max(self._global_best_history):
        self._last_improvement_iter = iteration
      self._global_best_history.append(global_best_score)

      # Notify database of no-improvement count for adaptive temperature
      no_improve = iteration - self._last_improvement_iter
      self._database.notify_no_improve(no_improve)
      
      # Record per-instance score and gap history (best so far each iteration)
      best_program_info = self._database.get_global_best_program()
      if best_program_info:
        best_island_id = best_program_info[1]
        scores_per_test = self._database._best_scores_per_test_per_island[best_island_id]
        if scores_per_test:
          score_dict: dict[str, float] = {}
          gap_dict: dict[str, float] = {}
          for instance, score in scores_per_test.items():
            instance_id = getattr(instance, "instance_id", str(instance))
            score_dict[instance_id] = float(score)
            optimal = getattr(instance, "optimal_tour_length", None)
            if optimal is not None and optimal > 0:
              tour_length = -float(score)
              gap_dict[instance_id] = (tour_length - optimal) / optimal * 100
          self._per_instance_score_history.append(score_dict)
          self._per_instance_gap_history.append(gap_dict)
        else:
          # No valid scores yet, carry forward previous or append empty
          prev_scores = self._per_instance_score_history[-1] if self._per_instance_score_history else {}
          prev_gaps = self._per_instance_gap_history[-1] if self._per_instance_gap_history else {}
          self._per_instance_score_history.append(prev_scores)
          self._per_instance_gap_history.append(prev_gaps)
      else:
        prev_scores = self._per_instance_score_history[-1] if self._per_instance_score_history else {}
        prev_gaps = self._per_instance_gap_history[-1] if self._per_instance_gap_history else {}
        self._per_instance_score_history.append(prev_scores)
        self._per_instance_gap_history.append(prev_gaps)
      
      # Calculate additional stats
      num_islands = len(self._database._islands)
      active_islands = sum(1 for score in self._database._best_score_per_island if score > -float('inf'))
      elapsed_seconds = int(time.time() - self._start_time)
      
      # Check if this iteration found a new global best
      new_global_best = (
          len(self._global_best_history) <= 1 or
          global_best_score > self._global_best_history[-2]
      )

      logging.info(
          "Iteration: %d | Total Tokens: %d | Best Score (Island %d): %s | "
          "Global Best: %s%s | Active Islands: %d/%d | Elapsed: %ds | Resets: %d | "
          "Version: %d",
          iteration, self._llm.total_tokens_used, prompt.island_id, best_score,
          global_best_score, " (NEW!)" if new_global_best else "",
          active_islands, num_islands, elapsed_seconds,
          self._database._reset_count, prompt.version_generated,
      )
      
      # Efficiency metrics (console output)
      eff = self._database.get_efficiency_stats()
      total_tokens = self._llm.total_tokens_used
      full_evals = eff["full_evaluations"]
      score_pt = round(global_best_score / total_tokens, 6) if total_tokens > 0 else "N/A"
      score_pe = round(global_best_score / full_evals, 6) if full_evals > 0 else "N/A"
      logging.info(
          "Efficiency: samples_attempted=%d | full_evaluations=%d | skipped=%s | "
          "score_per_token=%s | score_per_evaluation=%s",
          eff["samples_attempted"], full_evals, eff["skipped"], score_pt, score_pe,
      )
      
      # Cluster stats
      logging.info("Cluster Stats:")
      for i, island in enumerate(self._database._islands):
        if not island._clusters:
          logging.info("  Island %d: 0 clusters", i)
          continue
        logging.info("  Island %d: %d clusters", i, len(island._clusters))
        for sig, cluster in island._clusters.items():
          logging.info("    Cluster %s (Score: %s): %d programs", sig, cluster.score, len(cluster._programs))
      
      # Early stopping check (-1 disables)
      if self._early_stop_patience != -1 and iteration - self._last_improvement_iter >= self._early_stop_patience:
        logging.info(
            "Early stopping: No improvement for %d iterations (since iter %d)",
            self._early_stop_patience, self._last_improvement_iter,
        )
        break

  def generate_summary_report(self) -> Path | None:
    """Generates and saves experiment summary to result_dir with timestamp."""
    if not self._global_best_history:
      logging.warning("No iteration data to report.")
      return None
    
    unix_timestamp = int(time.time())
    out_dir = Path(self._result_dir) / f"{self._problem}_{unix_timestamp}"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    # 1. Save score progression plot (per-instance curves + average)
    try:
      import matplotlib
      matplotlib.use("Agg")
      import matplotlib.pyplot as plt
      from matplotlib.ticker import MaxNLocator
      
      plt.figure(figsize=(10, 6))
      iters = list(range(1, len(self._global_best_history) + 1))
      instance_ids = sorted(set(
        iid for h in self._per_instance_score_history for iid in h.keys()
      ))
      if instance_ids:
        for instance_id in instance_ids:
          scores = [
            h.get(instance_id) for h in self._per_instance_score_history
            if h.get(instance_id) is not None
          ]
          if scores:
            # Align length: use last known value for early iters without data
            full_scores = []
            last = None
            for h in self._per_instance_score_history:
              v = h.get(instance_id)
              last = v if v is not None else last
              full_scores.append(last)
            plt.plot(iters, full_scores, "-", linewidth=1, label=instance_id, alpha=0.8)
        avg_scores = [
          sum(h.values()) / len(h) if h else None
          for h in self._per_instance_score_history
        ]
        last_avg = None
        full_avg = []
        for v in avg_scores:
          last_avg = v if v is not None else last_avg
          full_avg.append(last_avg)
        if any(x is not None for x in full_avg):
          plt.plot(iters, full_avg, "k--", linewidth=2, label="Average")
        plt.legend(loc="best", fontsize=8)
      else:
        plt.plot(iters, self._global_best_history, "b-", linewidth=1, label="Global Best")
      plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
      plt.gca().set_xticks(iters)
      plt.xlabel("Iteration")
      plt.ylabel("Score")
      plt.title("Score Progression")
      plt.grid(True, alpha=0.3)
      plt.tight_layout()
      plt.savefig(out_dir / "score_progression.png", dpi=150)
      plt.close()
    except ImportError:
      logging.warning("matplotlib not installed, skipping score plot.")
    
    # 2. Build final_data and compute optimal comparison
    global_best = self._database.get_global_best_score()
    if global_best == -float("inf") and self._global_best_history:
        global_best = max(self._global_best_history)
    best_program_info = self._database.get_global_best_program()
    
    llm_model = os.getenv("LLM_MODEL", "arcee-ai/trinity-large-preview:free")
    final_data = {
        "total_iterations": self._total_iterations,
        "total_tokens": self._llm.total_tokens_used,
        "llm_model": llm_model,
        "global_best_score": global_best,
        "elapsed_seconds": int(time.time() - self._start_time),
        "score_history": self._global_best_history,
    }
    if best_program_info:
      final_data["best_island_id"] = best_program_info[1]
    
    # Compute optimal comparison (gap vs known optimum) for TSPLib instances
    optimal_comparison: list[dict] = []
    if best_program_info:
      best_island_id = best_program_info[1]
      scores_per_test = self._database._best_scores_per_test_per_island[best_island_id]
      if scores_per_test:
        for instance, score in scores_per_test.items():
          optimal = getattr(instance, "optimal_tour_length", None)
          if optimal is not None and optimal > 0:
            tour_length = -float(score)
            gap_pct = (tour_length - optimal) / optimal * 100
            optimal_comparison.append({
                "instance_id": instance.instance_id,
                "tour_length": round(tour_length, 6),
                "optimal": optimal,
                "gap_pct": round(gap_pct, 4),
            })
    if optimal_comparison:
      final_data["optimal_comparison"] = optimal_comparison
      final_data["avg_gap_pct"] = round(
          sum(c["gap_pct"] for c in optimal_comparison) / len(optimal_comparison), 4
      )
    
    # Per-instance score and gap history (all evaluate scores)
    instance_ids = sorted(set(
      iid for h in self._per_instance_score_history for iid in h.keys()
    ))
    if instance_ids:
      final_data["score_history_per_instance"] = {
        iid: [
          round(h.get(iid), 6) if h.get(iid) is not None else None
          for h in self._per_instance_score_history
        ]
        for iid in instance_ids
      }
      avg_score_history = []
      for h in self._per_instance_score_history:
        if h:
          avg_score_history.append(round(sum(h.values()) / len(h), 6))
        else:
          avg_score_history.append(None)
      final_data["avg_score_history"] = avg_score_history
    if self._per_instance_gap_history and any(h for h in self._per_instance_gap_history):
      gap_instance_ids = sorted(set(
        iid for h in self._per_instance_gap_history for iid in h.keys()
      ))
      final_data["gap_history_per_instance"] = {
        iid: [
          round(h.get(iid), 4) if h.get(iid) is not None else None
          for h in self._per_instance_gap_history
        ]
        for iid in gap_instance_ids
      }
      avg_gap_history = []
      for h in self._per_instance_gap_history:
        if h:
          avg_gap_history.append(round(sum(h.values()) / len(h), 4))
        else:
          avg_gap_history.append(None)
      final_data["avg_gap_history"] = avg_gap_history
    
    # Efficiency metrics
    eff = self._database.get_efficiency_stats()
    total_tokens = self._llm.total_tokens_used
    full_evaluations = eff["full_evaluations"]
    final_data["efficiency"] = {
        "samples_attempted": eff["samples_attempted"],
        "full_evaluations": full_evaluations,
        "skipped": eff["skipped"],
        "score_per_token": (
            round(global_best / total_tokens, 6) if total_tokens > 0 else None
        ),
        "score_per_evaluation": (
            round(global_best / full_evaluations, 6) if full_evaluations > 0 else None
        ),
    }
    
    # 2b. Save optimal comparison chart (when we have optimal_comparison data)
    if optimal_comparison:
      try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        from matplotlib.ticker import MaxNLocator
        ids = [c["instance_id"] for c in optimal_comparison]
        gaps = [c["gap_pct"] for c in optimal_comparison]
        avg_gap = final_data.get("avg_gap_pct", sum(gaps) / len(gaps))
        plt.figure(figsize=(8, 5))
        plt.bar(range(len(ids)), gaps, color="steelblue", edgecolor="navy", alpha=0.8)
        plt.axhline(y=avg_gap, color="red", linestyle="--", linewidth=1, label=f"Avg: {avg_gap:.2f}%")
        plt.xticks(range(len(ids)), ids, rotation=45, ha="right")
        plt.gca().yaxis.set_major_locator(MaxNLocator(integer=True))
        plt.xlabel("Instance")
        plt.ylabel("Gap (%)")
        plt.title("Optimal Comparison: Gap vs Known Optimum")
        plt.legend()
        plt.grid(True, alpha=0.3, axis="y")
        plt.tight_layout()
        plt.savefig(out_dir / "optimal_comparison.png", dpi=150)
        plt.close()
      except ImportError:
        logging.warning("matplotlib not installed, skipping optimal comparison plot.")
    
    # 2c. Save gap progression plot (per-instance curves + average)
    if self._per_instance_gap_history and any(h for h in self._per_instance_gap_history):
      try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        from matplotlib.ticker import MaxNLocator
        iters = list(range(1, len(self._per_instance_gap_history) + 1))
        instance_ids = sorted(set(
          iid for h in self._per_instance_gap_history for iid in h.keys()
        ))
        if instance_ids:
          plt.figure(figsize=(10, 6))
          for instance_id in instance_ids:
            full_gaps = []
            last = None
            for h in self._per_instance_gap_history:
              v = h.get(instance_id)
              last = v if v is not None else last
              full_gaps.append(last)
            if any(x is not None for x in full_gaps):
              plt.plot(iters, full_gaps, "-", linewidth=1, label=instance_id, alpha=0.8)
          avg_gaps = [
            sum(h.values()) / len(h) if h else None
            for h in self._per_instance_gap_history
          ]
          last_avg = None
          full_avg = []
          for v in avg_gaps:
            last_avg = v if v is not None else last_avg
            full_avg.append(last_avg)
          if any(x is not None for x in full_avg):
            plt.plot(iters, full_avg, "k--", linewidth=2, label="Average")
          plt.legend(loc="best", fontsize=8)
          plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
          plt.gca().set_xticks(iters)
          plt.xlabel("Iteration")
          plt.ylabel("Gap (%)")
          plt.title("Gap Progression vs Known Optimum")
          plt.grid(True, alpha=0.3)
          plt.tight_layout()
          plt.savefig(out_dir / "gap_progression.png", dpi=150)
          plt.close()
      except ImportError:
        logging.warning("matplotlib not installed, skipping gap progression plot.")
    
    with open(out_dir / "final_results.json", "w", encoding="utf-8") as f:
      json.dump(final_data, f, indent=2, ensure_ascii=False)
    
    # 3. Save best program code
    if best_program_info and best_program_info[0] is not None and self._template is not None and self._function_to_evolve:
      best_func, _ = best_program_info
      program = copy.deepcopy(self._template)
      evolved = program.get_function(self._function_to_evolve)
      evolved.body = best_func.body
      with open(out_dir / "best_program.py", "w", encoding="utf-8") as f:
        f.write(str(program))
    
    # 4. Save experiment summary markdown
    with open(out_dir / "experiment_summary.md", "w", encoding="utf-8") as f:
      f.write("# Experiment Summary\n\n")
      f.write(f"- **LLM Model**: {llm_model}\n")
      f.write(f"- **Total Iterations**: {self._total_iterations}\n")
      f.write(f"- **Total Tokens**: {self._llm.total_tokens_used}\n")
      f.write(f"- **Global Best Score**: {global_best}\n")
      f.write(f"- **Elapsed Time**: {final_data['elapsed_seconds']}s\n")
      if best_program_info:
        f.write(f"- **Best Island**: {best_program_info[1]}\n")
      f.write("\n## Score Progression\n\n")
      f.write("See `score_progression.png` for the visualization.\n")
      eff = self._database.get_efficiency_stats()
      total_tokens = self._llm.total_tokens_used
      full_evaluations = eff["full_evaluations"]
      f.write("\n## Sample Efficiency\n\n")
      f.write("| Metric | Value |\n")
      f.write("|--------|-------|\n")
      f.write(f"| Samples Attempted | {eff['samples_attempted']} |\n")
      f.write(f"| Full Evaluations | {full_evaluations} |\n")
      score_pt = (
          round(global_best / total_tokens, 6) if total_tokens > 0 else "N/A"
      )
      score_pe = (
          round(global_best / full_evaluations, 6) if full_evaluations > 0 else "N/A"
      )
      f.write(f"| Score per Token | {score_pt} |\n")
      f.write(f"| Score per Evaluation | {score_pe} |\n")
      if eff["skipped"]:
        f.write("\n### Skipped Breakdown\n\n")
        for reason, count in sorted(eff["skipped"].items()):
          f.write(f"- {reason}: {count}\n")
      if optimal_comparison:
        f.write("\n## Optimal Comparison\n\n")
        f.write("| Instance | Tour Length | Optimal | Gap (%) |\n")
        f.write("|----------|-------------|---------|----------|\n")
        for c in optimal_comparison:
          f.write(f"| {c['instance_id']} | {c['tour_length']} | {c['optimal']} | {c['gap_pct']} |\n")
        f.write(f"\nAverage gap: {final_data['avg_gap_pct']}%\n\n")
        f.write("See `optimal_comparison.png` and `gap_progression.png` for the visualizations.\n")
    
    logging.info("Results saved to %s (model: %s)", out_dir, llm_model)
    
    # Print all efficiency metrics to console (eff, total_tokens, etc. from above)
    logging.info("=== Sample Efficiency (Final) ===")
    logging.info("  Samples Attempted: %d", eff["samples_attempted"])
    logging.info("  Full Evaluations: %d", full_evaluations)
    logging.info("  Score per Token: %s", score_pt)
    logging.info("  Score per Evaluation: %s", score_pe)
    if eff["skipped"]:
      logging.info("  Skipped Breakdown:")
      for reason, count in sorted(eff["skipped"].items()):
        logging.info("    - %s: %d", reason, count)
    else:
      logging.info("  Skipped Breakdown: (none)")
    
    return out_dir



def _extract_function_names(specification: str) -> tuple[str, str]:
  from funsearch.implementation import code_manipulation
  run_functions = list(
      code_manipulation.yield_decorated(specification, 'funsearch', 'run'))
  if len(run_functions) != 1:
    raise ValueError('Expected 1 function decorated with `@funsearch.run`.')
  evolve_functions = list(
      code_manipulation.yield_decorated(specification, 'funsearch', 'evolve'))
  if len(evolve_functions) != 1:
    raise ValueError('Expected 1 function decorated with `@funsearch.evolve`.')
  return evolve_functions[0], run_functions[0]


def main_adapted(specification: str, inputs, config):
  """Same wiring as implementation/funsearch.main but uses Adapted* classes (inherit upstream)."""
  from pathlib import Path
  from funsearch.implementation import code_manipulation
  from implementation.functional_dedup import FunctionalDedupCache

  function_to_evolve, function_to_run = _extract_function_names(specification)
  template = code_manipulation.text_to_program(specification)
  database = AdaptedProgramsDatabase(
      config.programs_database, template, function_to_evolve, config_full=config)

  dedup_cache = (
      FunctionalDedupCache(tier1_only=config.dedup_tier1_only)
      if config.functional_dedup
      else None
  )

  evaluators = []
  for _ in range(config.num_evaluators):
    evaluators.append(AdaptedEvaluator(
        database,
        template,
        function_to_evolve,
        function_to_run,
        inputs,
        config=config,
        dedup_cache=dedup_cache,
    ))
  initial = template.get_function(function_to_evolve).body
  evaluators[0].analyse(initial, island_id=None, version_generated=None)

  result_dir = Path(config.result_dir)
  result_dir.mkdir(parents=True, exist_ok=True)

  samplers = [
      AdaptedSampler(
          database,
          evaluators,
          config.samples_per_prompt,
          config.iterations,
          config.goal_description,
          early_stop_patience=config.early_stop_patience,
          result_dir=str(result_dir),
          problem=config.problem,
          template=template,
          function_to_evolve=function_to_evolve,
          config=config,
      )
      for _ in range(config.num_samplers)
  ]
  for s in samplers:
    s.sample()
  if samplers:
    samplers[0].generate_summary_report()


In [ ]:
# Build Config + run main_adapted (uses SPECIFICATION and Config variables from above)
import dataclasses
from dotenv import load_dotenv
from implementation import config as ic
from implementation.tsp_utils import prepare_test_inputs

load_dotenv()

full_config = ic.Config(
    programs_database=ic.ProgramsDatabaseConfig(
        functions_per_prompt=programs_database_functions_per_prompt,
        num_islands=programs_database_num_islands,
        reset_period=programs_database_reset_period,
        cluster_sampling_temperature_init=programs_database_cluster_sampling_temperature_init,
        cluster_sampling_temperature_period=programs_database_cluster_sampling_temperature_period,
    ),
    num_samplers=num_samplers,
    num_evaluators=num_evaluators,
    samples_per_prompt=samples_per_prompt,
    iterations=iterations,
    goal_description=goal_description,
    early_stop_patience=early_stop_patience,
    result_dir=result_dir,
    problem=problem,
    progressive_eval=progressive_eval,
    stage1_timeout=stage1_timeout,
    stage1_score_threshold_pct=stage1_score_threshold_pct,
    functional_dedup=functional_dedup,
    dedup_tier1_only=dedup_tier1_only,
    adaptive_sampling=adaptive_sampling,
    min_samples_per_prompt=min_samples_per_prompt,
    max_samples_per_prompt=max_samples_per_prompt,
    reduce_after_no_improve=reduce_after_no_improve,
    weighted_island_sampling=weighted_island_sampling,
    island_sampling_temperature=island_sampling_temperature,
    feedback_in_prompt=feedback_in_prompt,
    adaptive_temperature=adaptive_temperature,
    reheat_after_no_improve=reheat_after_no_improve,
    reheat_temperature_multiplier=reheat_temperature_multiplier,
)

full_config = dataclasses.replace(
    full_config,
    problem="tsp",
    goal_description=(
        "minimize the total TSP tour length. The evaluate function returns "
        "negative tour length (higher is better)."
    ),
)

test_inputs = prepare_test_inputs(tsplib_paths=None, random_specs=[(12, 42), (15, 99)])

main_adapted(SPECIFICATION, test_inputs, full_config)
